# MLflow 概要 — 4つのコンポーネント

このノートブックでは、MLflow の主要な4つのコンポーネントの概念と相互関係を学びます。

## 前提条件

- **Python**: 3.11
- **MLflow**: >= 2.8.0
- **conda 仮想環境**: `mlflow-tutorial` がアクティブであること

```bash
conda activate mlflow-tutorial
```

## 学習目標

- MLflow の4コンポーネント（Tracking / Models / Model Registry / Projects）の役割を理解する
- 各コンポーネントがどのように連携するかを理解する
- 実際にコードを動かして MLflow Tracking の基本的な使い方を習得する

---

## 1. MLflow Tracking

### 概念

**MLflow Tracking** は、ML 実験のパラメータ・メトリクス・アーティファクトを記録・管理する仕組みです。

| 記録対象 | 説明 | 例 |
|---------|------|----|
| **パラメータ** | 固定のハイパーパラメータ | `learning_rate=0.01` |
| **メトリクス** | ステップごとに変化する数値 | `loss`, `accuracy` |
| **アーティファクト** | ファイルやモデルなど任意のオブジェクト | モデルファイル, 画像 |
| **タグ** | 実験を識別するラベル | `team=research` |

### 主要 API

```python
import mlflow

# 実験を開始する
with mlflow.start_run(run_name="my_run") as run:
    # パラメータを記録
    mlflow.log_param("learning_rate", 0.01)

    # メトリクスをステップ付きで記録
    mlflow.log_metric("loss", 0.5, step=0)
    mlflow.log_metric("loss", 0.3, step=1)

    # アーティファクト（ファイル）を記録
    mlflow.log_artifact("output.png")
```

### Tracking Server の構成

- **ローカルファイルシステム**: デフォルトで `./mlruns` に保存（開発時に便利）
- **リモートサーバー**: `mlflow server` で起動したサーバーに接続（チーム共有時）
- **クラウドストレージ**: S3, GCS, Azure Blob Storage などと連携可能

---

## 2. MLflow Models

### 概念

**MLflow Models** は、様々な ML フレームワークのモデルを **標準フォーマット** でパッケージ化する仕組みです。  
「フレーバー（Flavor）」という概念により、異なるフレームワークのモデルを統一された方法で扱えます。

### フレーバー（Flavor）

| フレーバー | 説明 | 主な用途 |
|-----------|------|----------|
| **sklearn** | scikit-learn モデル専用 | 分類・回帰・クラスタリング |
| **pytorch** | PyTorch モデル専用 | ディープラーニング |
| **tensorflow** | TensorFlow/Keras モデル専用 | ディープラーニング |
| **pyfunc** | 汎用 Python 関数ラッパー | 前処理込みのカスタムモデル |

### 主要 API

```python
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression

model = LinearRegression()
# ... 学習 ...

with mlflow.start_run():
    # モデルをアーティファクトとして記録
    mlflow.sklearn.log_model(model, artifact_path="model")

# 記録したモデルを汎用 pyfunc フレーバーとしてロード
loaded_model = mlflow.pyfunc.load_model("runs:/<run_id>/model")
predictions = loaded_model.predict(X_test)
```

### モデル URI の形式

```
runs:/<run_id>/<artifact_path>      # ランから直接
models:/<model_name>/<version>      # レジストリのバージョン番号
models:/<model_name>@<alias>        # レジストリのエイリアス
```

---

## 3. MLflow Model Registry

### 概念

**MLflow Model Registry** は、モデルの **ライフサイクル管理** を行う仕組みです。  
実験中に記録した多数のモデルの中から、本番利用するモデルを選択・管理できます。

### 主な機能

- **登録（Register）**: Tracking に記録されたモデルをレジストリに登録
- **バージョン管理**: 同一モデル名で複数バージョンを管理
- **エイリアス（Alias）**: バージョンに名前をつけて参照しやすくする

### エイリアスの慣例

| エイリアス | 意味 |
|-----------|------|
| **champion** | 現在本番で使用中のモデル |
| **challenger** | champion と比較検証中のモデル |
| **archived** | 過去に使用されたが現在は非推奨のモデル |

### 主要 API

```python
import mlflow
from mlflow import MlflowClient

# ランのモデルをレジストリに登録
mlflow.register_model(
    model_uri="runs:/<run_id>/model",
    name="my_model"
)

client = MlflowClient()

# エイリアスを設定（バージョン 1 を champion に）
client.set_registered_model_alias(
    name="my_model",
    alias="champion",
    version="1"
)

# エイリアスでモデルをロード
model = mlflow.pyfunc.load_model("models:/my_model@champion")
```

---

## 4. MLflow Projects

### 概念

**MLflow Projects** は、ML コードを **再現可能な形でパッケージ化** する仕組みです。  
`MLproject` ファイルに実行環境・エントリーポイント・パラメータを定義することで、  
異なる環境でも同じコードを同じように実行できます。

### MLproject ファイルの例

```yaml
# MLproject
name: my_project

conda_env: conda.yaml  # または python_env: python_env.yaml

entry_points:
  main:
    parameters:
      learning_rate: {type: float, default: 0.01}
      epochs: {type: int, default: 10}
    command: "python train.py --lr {learning_rate} --epochs {epochs}"
```

### 実行方法

```bash
mlflow run . -P learning_rate=0.05 -P epochs=20
```

> **このチュートリアルについて**: このチュートリアルでは Projects は主題ではありません。  
> 各ノートブックはそのまま実行可能で、再現性は conda 環境と `environment.yml` によって管理します。

---

## 5. コンポーネントの相互関係

4つのコンポーネントは以下のように連携しています:

```
┌─────────────────────────────────────────────────────────────┐
│                    ML 開発ワークフロー                        │
│                                                              │
│  ┌──────────────────┐     記録      ┌──────────────────────┐│
│  │  MLflow Projects │ ──────────→  │   MLflow Tracking    ││
│  │（再現可能な実行） │              │（実験・メトリクス管理）││
│  └──────────────────┘              └──────────┬───────────┘│
│                                               │ モデルを    │
│                                               │ アーティ   │
│                                               │ ファクト   │
│                                               ↓ として記録  │
│                                    ┌──────────────────────┐│
│                                    │    MLflow Models     ││
│                                    │（標準フォーマット化）  ││
│                                    └──────────┬───────────┘│
│                                               │ レジストリ  │
│                                               │ に登録      │
│                                               ↓             │
│                                    ┌──────────────────────┐│
│                                    │ MLflow Model Registry││
│                                    │（ライフサイクル管理） ││
│                                    │  champion / challenger││
│                                    └──────────────────────┘│
└─────────────────────────────────────────────────────────────┘
```

**データの流れ**:
1. **Projects** が実行環境を定義し、再現可能な形でコードを実行
2. 実行中に **Tracking** がパラメータ・メトリクス・モデルを記録
3. 記録されたモデルは **Models** 標準フォーマットでパッケージ化
4. 優れたモデルを **Model Registry** に登録してライフサイクルを管理

---

## 6. MLflow UI の起動と確認

### ローカル Tracking Server UI の起動

ターミナルで以下のコマンドを実行します:

```bash
# conda 環境をアクティブにした状態で実行
conda activate mlflow-tutorial
mlflow ui
```

デフォルトで `http://127.0.0.1:5000` で UI が起動します。

### UI で確認できること

| 画面 | 説明 |
|------|------|
| **Experiments** | 実験一覧・ラン一覧の確認 |
| **Run 詳細** | パラメータ・メトリクス・アーティファクトの確認 |
| **メトリクスグラフ** | ステップごとのメトリクス推移をグラフで可視化 |
| **Models** | Model Registry に登録されたモデルの管理 |

### 接続先の確認

In [ ]:
import mlflow

# 現在の Tracking URI を確認
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

---

## 7. 実際に動かしてみる

MLflow Tracking の基本操作を実際に動かして確認します。  
ローカルファイルシステム（`./mlruns`）に実験結果を記録します。

In [ ]:
import mlflow
import numpy as np

# 乱数シードを固定（再現性のため）
np.random.seed(42)

# ローカルファイルシステムに記録
mlflow.set_tracking_uri("./mlruns")
mlflow.set_experiment("01_concepts_demo")

with mlflow.start_run(run_name="概念デモ") as run:
    # パラメータ記録
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("epochs", 10)

    # メトリクス記録（エポックごと）
    for epoch in range(10):
        loss = np.exp(-epoch * 0.1) + np.random.normal(0, 0.01)
        mlflow.log_metric("loss", loss, step=epoch)

    print(f"Run ID: {run.info.run_id}")
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")

### 実行結果の確認

上のセルを実行すると:

1. `./mlruns/` ディレクトリが作成され、実験データが保存されます
2. `Run ID` が表示されます（各ランを一意に識別するUUID）
3. ターミナルで `mlflow ui` を実行すると、UI でメトリクスのグラフを確認できます

**確認のポイント**:
- `loss` メトリクスがエポックとともに減少していることをグラフで確認
- `learning_rate`, `epochs` パラメータが記録されていることを確認

---

## まとめ

### 4コンポーネントの要点

| コンポーネント | 役割 | キーワード |
|--------------|------|----------|
| **Tracking** | 実験の記録・管理 | パラメータ / メトリクス / アーティファクト |
| **Models** | モデルの標準フォーマット化 | フレーバー / pyfunc / モデルURI |
| **Model Registry** | モデルのライフサイクル管理 | バージョン / エイリアス / champion |
| **Projects** | 実行環境の定義・再現 | MLproject / conda_env / エントリーポイント |

### 次に読むべきノートブック

- **[scikit-learn サンプル](../02_sklearn/sklearn_quickstart.ipynb)**  
  sklearn モデルの学習から Model Registry 登録までの一連のフローを実践

- **[PyTorch サンプル](../03_pytorch/pytorch_quickstart.ipynb)**  
  PyTorch モデルのトレーニングと MLflow Tracking の統合を実践

---

> **参考リンク**:
> - [MLflow 公式ドキュメント](https://mlflow.org/docs/latest/index.html)
> - [MLflow Tracking コンセプト](https://mlflow.org/docs/latest/tracking.html)
> - [MLflow Model Registry](https://mlflow.org/docs/latest/model-registry.html)